# 03 - Bayesian VAR (BVAR) with Minnesota Prior

This notebook covers **Bayesian Vector Autoregression** models with emphasis on the
Minnesota prior for macroeconomic forecasting and structural analysis.

## Topics covered

1. Motivation for Bayesian estimation of VARs
2. The Minnesota prior (Litterman, 1986)
3. Hyperparameter specification and sensitivity
4. BVAR vs frequentist VAR comparison
5. Bayesian forecasting with credible intervals
6. Structural IRFs from BVAR
7. Exercises

---

## Why Bayesian VAR?

Classical VAR models suffer from the **curse of dimensionality**: a VAR(p) with $K$
variables has $K^2 p + K$ parameters. With 4 variables and 4 lags, that is
$4^2 \times 4 + 4 = 68$ parameters — more than many macro datasets can reliably estimate.

**Bayesian shrinkage** addresses this by imposing **prior beliefs** that regularize
the estimates:

$$p(\beta, \Sigma \mid Y) \propto p(Y \mid \beta, \Sigma) \cdot p(\beta, \Sigma)$$

The **Minnesota prior** (Litterman, 1986; Doan, Litterman & Sims, 1984) is the most
widely used prior for VARs. Its key idea:

> *Each variable follows a random walk (or AR(1)) with cross-variable effects shrunk toward zero.*

This is encoded via the prior variance on the coefficients:

$$\text{Var}(A_{jk}^{(l)}) = \begin{cases} \left(\frac{\lambda_1}{l^{\lambda_3}}\right)^2 & \text{if } j = k \text{ (own lag)} \\ \left(\frac{\lambda_1 \cdot \lambda_2}{l^{\lambda_3}}\right)^2 \cdot \frac{\hat{\sigma}_j^2}{\hat{\sigma}_k^2} & \text{if } j \neq k \text{ (cross-variable)} \end{cases}$$

where:
- $\lambda_1$: **overall tightness** — lower values = stronger shrinkage toward prior
- $\lambda_2$: **cross-variable shrinkage** — controls how much other variables' lags matter (0 to 1)
- $\lambda_3$: **lag decay** — higher values = more aggressive shrinkage at longer lags

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

from chronobox.models import VAR, BayesianVAR

sys.path.insert(0, os.path.join("..", "utils"))
from plot_helpers import plot_structural_irf

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.facecolor"] = "white"
np.set_printoptions(precision=4, suppress=True)

print("All imports loaded successfully.")

## 1. Loading Data

We use the same US macro quarterly dataset with 4 variables: GDP growth, inflation,
federal funds rate, and unemployment.

In [ ]:
# Load US macro data
data_path = os.path.join("..", "data", "us_macro_quarterly.csv")
df = pd.read_csv(data_path, parse_dates=["date"])
df.set_index("date", inplace=True)

var_names = ["gdp", "inflation", "fed_funds", "unemployment"]
endog = df[var_names].values

print(f"Dataset: {endog.shape[0]} observations, {endog.shape[1]} variables")
print(f"Variables: {var_names}")
print(f"\nWith 4 variables and 4 lags:")
print(f"  OLS parameters: {4**2 * 4 + 4} = K^2*p + K")
print(f"  Observations: {endog.shape[0]}")
print(f"  Parameters/observations ratio: {(4**2 * 4 + 4) / endog.shape[0]:.2f}")
print(f"\n=> Bayesian shrinkage helps when this ratio approaches or exceeds ~0.3")

## 2. BVAR with Minnesota Prior — Baseline

We start with standard Minnesota prior hyperparameters:
- $\lambda_1 = 0.1$ (moderate tightness)
- $\lambda_2 = 0.5$ (moderate cross-variable shrinkage)
- $\lambda_3 = 1.0$ (linear lag decay)
- $\delta = 1.0$ (unit root prior: each variable's own first lag has prior mean = 1)

In [ ]:
# Fit BVAR with Minnesota prior
bvar = BayesianVAR(
    lags=4,
    prior="minnesota",
    lambda_1=0.1,     # overall tightness
    lambda_2=0.5,     # cross-variable shrinkage
    lambda_3=1.0,     # lag decay
    delta=1.0,        # unit root prior mean
)

bvar_results = bvar.fit(endog, n_draws=5000, burnin=1000, seed=42)

print(f"BVAR({bvar_results.lags}) with Minnesota prior fitted")
print(f"Variables: {bvar_results.k_vars}, Observations: {bvar_results.n_obs}")
print(f"Posterior draws retained: {bvar_results.coefs_draws.shape[0]}")
print(f"Log marginal likelihood: {bvar_results.log_marginal_likelihood:.2f}")
print(f"\nPosterior mean coefficients (lag 1):")
print(pd.DataFrame(bvar_results.coefs_mean[0].round(4), 
                    index=var_names, columns=var_names))

## 3. BVAR vs Frequentist VAR Comparison

Let us compare the posterior mean coefficients from BVAR with the OLS estimates
from a standard VAR. The Minnesota prior should **shrink** the coefficients toward
the random walk prior, especially for cross-variable and higher-lag coefficients.

In [ ]:
# Fit frequentist VAR for comparison
var_freq = VAR(lags=4, trend="c")
var_freq_res = var_freq.fit(endog)

# Compare lag 1 coefficients
print("=== LAG 1 COEFFICIENTS ===")
print(f"\nOLS (frequentist):")
ols_coefs_l1 = var_freq_res.coefs[0] if hasattr(var_freq_res, 'coefs') else var_freq_res.params[:4, :].T
print(pd.DataFrame(ols_coefs_l1.round(4), index=var_names, columns=var_names))

print(f"\nBVAR (Minnesota posterior mean):")
print(pd.DataFrame(bvar_results.coefs_mean[0].round(4), index=var_names, columns=var_names))

# Show shrinkage effect
print(f"\n=== SHRINKAGE ANALYSIS ===")
for lag in range(4):
    ols_l = var_freq_res.coefs[lag] if hasattr(var_freq_res, 'coefs') else None
    bvar_l = bvar_results.coefs_mean[lag]
    if ols_l is not None:
        # Off-diagonal elements (cross-variable effects)
        mask = ~np.eye(4, dtype=bool)
        ols_off = np.abs(ols_l[mask]).mean()
        bvar_off = np.abs(bvar_l[mask]).mean()
        shrink = 1 - bvar_off / ols_off if ols_off > 0 else 0
        print(f"Lag {lag+1}: mean |cross-variable coef| OLS={ols_off:.4f}, "
              f"BVAR={bvar_off:.4f}, shrinkage={shrink:.1%}")

In [ ]:
# Visual comparison: coefficient scatter plot (OLS vs BVAR)
ols_all = np.concatenate([var_freq_res.coefs[l].flatten() for l in range(4)])
bvar_all = np.concatenate([bvar_results.coefs_mean[l].flatten() for l in range(4)])

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.scatter(ols_all, bvar_all, alpha=0.5, s=30, color="steelblue")
lims = [min(ols_all.min(), bvar_all.min()) - 0.05, 
        max(ols_all.max(), bvar_all.max()) + 0.05]
ax.plot(lims, lims, "k--", linewidth=0.8, label="45-degree line")
ax.plot(lims, [0, 0], "r-", linewidth=0.5, alpha=0.5)
ax.plot([0, 0], lims, "r-", linewidth=0.5, alpha=0.5)
ax.set_xlabel("OLS Coefficients (Frequentist VAR)", fontsize=11)
ax.set_ylabel("Posterior Mean (BVAR Minnesota)", fontsize=11)
ax.set_title("Shrinkage Effect: BVAR Minnesota vs OLS", fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "bvar_shrinkage_scatter.png"), bbox_inches="tight")
plt.show()

print("Points below the 45-degree line are shrunk toward zero by the Minnesota prior.")

## 4. Hyperparameter Sensitivity

The Minnesota prior's behavior depends critically on the hyperparameters. Let us explore
how $\lambda_1$ (overall tightness) affects the estimates.

- $\lambda_1 \to 0$: strong shrinkage, BVAR approaches the prior (random walk)
- $\lambda_1 \to \infty$: no shrinkage, BVAR converges to OLS

In [ ]:
# Sensitivity to lambda_1 (overall tightness)
lambda_1_values = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
results_by_lambda = {}

for lam in lambda_1_values:
    bvar_tmp = BayesianVAR(lags=4, prior="minnesota", lambda_1=lam, lambda_2=0.5)
    res_tmp = bvar_tmp.fit(endog, n_draws=2000, burnin=500, seed=42)
    results_by_lambda[lam] = res_tmp
    print(f"lambda_1={lam:.2f}: log ML = {res_tmp.log_marginal_likelihood:.2f}")

# Find optimal lambda_1 by marginal likelihood
best_lambda = max(results_by_lambda, key=lambda l: results_by_lambda[l].log_marginal_likelihood)
print(f"\nOptimal lambda_1 (by marginal likelihood): {best_lambda}")

In [ ]:
# Plot IRFs for different tightness levels
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
horizons = np.arange(21)
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(lambda_1_values)))

for lam, color in zip(lambda_1_values, colors):
    irf_tmp = results_by_lambda[lam].irf(periods=20)
    # Plot response of GDP to fed_funds shock (monetary policy)
    for i, (ax, name) in enumerate(zip(axes.flat, var_names)):
        ax.plot(horizons, irf_tmp[:, i, 2], color=color, linewidth=1.2,
                label=f"λ₁={lam}" if i == 0 else None)

for i, (ax, name) in enumerate(zip(axes.flat, var_names)):
    ax.axhline(0, color="k", linewidth=0.5)
    ax.set_title(f"Response of {name} to monetary shock", fontsize=10)
    ax.set_xlabel("Quarters")
    ax.grid(True, alpha=0.3)

axes[0, 0].legend(fontsize=7, loc="best")
plt.suptitle("BVAR IRF Sensitivity to λ₁ (Overall Tightness)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "bvar_lambda1_sensitivity.png"), bbox_inches="tight")
plt.show()

## 5. Bayesian Forecasting with Credible Intervals

A major advantage of BVAR: forecasts naturally incorporate **parameter uncertainty**
through the posterior distribution. The predictive distribution is:

$$p(Y_{T+h} \mid Y) = \int p(Y_{T+h} \mid \beta, \Sigma, Y) \, p(\beta, \Sigma \mid Y) \, d\beta \, d\Sigma$$

This gives **credible intervals** (not confidence intervals) — they directly represent
the probability of the forecast lying within the band.

In [ ]:
# Bayesian forecast
forecast = bvar_results.forecast(steps=12)

print(f"Forecast keys: {list(forecast.keys())}")
print(f"Mean forecast shape: {forecast['mean'].shape}  (steps, K)")

# Plot forecasts with credible intervals
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
titles = ["GDP Growth (%)", "Inflation (%)", "Fed Funds Rate (%)", "Unemployment (%)"]
colors = ["steelblue", "firebrick", "darkgreen", "darkorange"]

# Show last 20 observations + 12-step forecast
n_hist = 20
hist_idx = np.arange(n_hist)
fc_idx = np.arange(n_hist, n_hist + 12)

for i, (ax, name, title, color) in enumerate(zip(axes.flat, var_names, titles, colors)):
    # Historical data
    ax.plot(hist_idx, endog[-n_hist:, i], color=color, linewidth=1.5, label="Observed")
    
    # Forecast mean
    ax.plot(fc_idx, forecast["mean"][:, i], color=color, linewidth=2, 
            linestyle="--", label="Forecast (mean)")
    
    # 68% credible interval
    ax.fill_between(fc_idx, forecast["lower_68"][:, i], forecast["upper_68"][:, i],
                    alpha=0.3, color=color, label="68% CI")
    
    # 95% credible interval
    ax.fill_between(fc_idx, forecast["lower_95"][:, i], forecast["upper_95"][:, i],
                    alpha=0.1, color=color, label="95% CI")
    
    ax.axvline(n_hist - 0.5, color="gray", linestyle=":", linewidth=1)
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=7, loc="best")

plt.suptitle("BVAR Minnesota: 12-Quarter Forecast with Credible Intervals", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "bvar_forecast.png"), bbox_inches="tight")
plt.show()

## 6. Structural IRFs from BVAR

We can compute structural IRFs from the BVAR using Cholesky identification on the
posterior mean, and also compute **IRF draws** across the posterior to get credible bands.

In [ ]:
# IRF from posterior mean
irf_bvar_mean = bvar_results.irf(periods=20, method="cholesky")
print(f"IRF (posterior mean) shape: {irf_bvar_mean.shape}")

# IRF draws for credible bands
irf_draws = bvar_results.irf_draws_compute(periods=20, method="cholesky")
print(f"IRF draws shape: {irf_draws.shape}  (n_draws, periods+1, K, K)")

# Compute credible bands
irf_median = np.median(irf_draws, axis=0)
irf_lower = np.quantile(irf_draws, 0.16, axis=0)
irf_upper = np.quantile(irf_draws, 0.84, axis=0)

# Plot structural IRFs with Bayesian credible bands
shock_names = ["GDP shock", "Inflation shock", "Monetary shock", "Unemp. shock"]
fig = plot_structural_irf(
    irf_median,
    variable_names=var_names,
    shock_names=shock_names,
    ci_lower=irf_lower,
    ci_upper=irf_upper,
    title="BVAR Minnesota: Structural IRF with 68% Credible Bands",
    figsize=(16, 12),
)
plt.savefig(os.path.join("..", "outputs", "bvar_structural_irf.png"), bbox_inches="tight")
plt.show()

### 6.1 BVAR vs Frequentist VAR: IRF Comparison

Let us overlay the BVAR and OLS-based IRFs for the monetary policy shock to see
how Bayesian shrinkage affects the estimated transmission mechanism.

In [ ]:
# Frequentist SVAR IRF for comparison
from chronobox.models import SVAR

svar_freq = SVAR(var_freq_res, method="cholesky")
results_freq = svar_freq.fit()
irf_freq = results_freq.irf(periods=20)

# Compare BVAR vs OLS for monetary shock
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, (ax, name, title) in enumerate(zip(axes.flat, var_names, titles)):
    # Frequentist
    ax.plot(horizons, irf_freq[:, i, 2], "r-", linewidth=1.8, label="VAR (OLS)")
    # BVAR (median + bands)
    ax.plot(horizons, irf_median[:, i, 2], "b-", linewidth=1.8, label="BVAR (median)")
    ax.fill_between(horizons, irf_lower[:, i, 2], irf_upper[:, i, 2],
                    alpha=0.2, color="blue")
    ax.axhline(0, color="k", linewidth=0.5)
    ax.set_title(f"{title}", fontsize=10)
    ax.set_xlabel("Quarters")
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=9)

plt.suptitle("Monetary Shock: BVAR Minnesota vs Frequentist VAR", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "bvar_vs_var_monetary_irf.png"), bbox_inches="tight")
plt.show()

**Key observations:**

- The BVAR IRFs are **smoother** than the OLS IRFs — this is the shrinkage effect
- The Bayesian credible bands quantify **parameter uncertainty** around the IRF
- If the OLS IRF lies within the BVAR credible bands, the two approaches are broadly consistent
- The BVAR is particularly valuable when the sample is short or the model is large

## 7. Model Selection via Marginal Likelihood

The **marginal likelihood** (or Bayesian evidence) allows comparison across priors and
hyperparameter settings. It naturally penalizes overfitting through the Bayesian Occam's razor.

In [ ]:
# Compare priors via marginal likelihood
priors_to_compare = {
    "Minnesota (λ₁=0.1)": {"prior": "minnesota", "lambda_1": 0.1, "lambda_2": 0.5},
    "Minnesota (λ₁=0.2)": {"prior": "minnesota", "lambda_1": 0.2, "lambda_2": 0.5},
    "Minnesota (tight, λ₁=0.01)": {"prior": "minnesota", "lambda_1": 0.01, "lambda_2": 0.5},
    "Normal-Wishart": {"prior": "normal_wishart"},
    "Flat (diffuse)": {"prior": "flat"},
}

comparison_results = {}
for name, kwargs in priors_to_compare.items():
    bvar_cmp = BayesianVAR(lags=4, **kwargs)
    res_cmp = bvar_cmp.fit(endog, n_draws=2000, burnin=500, seed=42)
    comparison_results[name] = res_cmp.log_marginal_likelihood
    print(f"{name:35s}: log ML = {res_cmp.log_marginal_likelihood:10.2f}")

# Plot marginal likelihood comparison
fig, ax = plt.subplots(figsize=(10, 5))
names = list(comparison_results.keys())
values = list(comparison_results.values())
bars = ax.barh(names, values, color="steelblue", alpha=0.8)
ax.set_xlabel("Log Marginal Likelihood", fontsize=11)
ax.set_title("Model Comparison via Marginal Likelihood", fontsize=13)
ax.grid(True, alpha=0.3, axis="x")

# Highlight best model
best_idx = np.argmax(values)
bars[best_idx].set_color("darkgreen")
bars[best_idx].set_alpha(1.0)

plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "bvar_model_comparison.png"), bbox_inches="tight")
plt.show()

## Exercicio

### Exercicio 1: Ajustar BVAR com diferentes priors

Compare tres especificacoes de BVAR:

1. **Minnesota apertado**: $\lambda_1 = 0.01$, $\lambda_2 = 0.1$ (forte shrinkage)
2. **Minnesota padrao**: $\lambda_1 = 0.1$, $\lambda_2 = 0.5$ (baseline)
3. **Minnesota frouxo**: $\lambda_1 = 1.0$, $\lambda_2 = 1.0$ (pouco shrinkage)

Para cada especificacao:
- Compute a previsao 8 passos a frente
- Plote os intervalos de credibilidade de 95%
- Compare a largura dos intervalos: qual prior produz intervalos mais estreitos?
- Use a marginal likelihood para selecionar o melhor modelo

In [ ]:
# --- Exercicio 1: Seu codigo aqui ---

# Dica:
# configs = {
#     "Tight": {"lambda_1": 0.01, "lambda_2": 0.1},
#     "Standard": {"lambda_1": 0.1, "lambda_2": 0.5},
#     "Loose": {"lambda_1": 1.0, "lambda_2": 1.0},
# }
# for name, params in configs.items():
#     bvar_ex = BayesianVAR(lags=4, prior="minnesota", **params)
#     res_ex = bvar_ex.fit(endog, n_draws=3000, burnin=500, seed=42)
#     fc = res_ex.forecast(steps=8)
#     print(f"{name}: log ML = {res_ex.log_marginal_likelihood:.2f}")

### Exercicio 2: BVAR com prior Sims-Zha

A prior Sims-Zha estende a Minnesota com duas restricoes adicionais:
- **Sum-of-coefficients** ($\mu_5$): impoe que a soma dos coeficientes de cada variavel
  e proxima de 1 (unit root prior)
- **Co-persistence** ($\mu_6$): impoe que todas as variaveis compartilham uma tendencia
  estocastica comum

1. Ajuste um BVAR com `prior="sims_zha"`, `mu_5=1.0`, `mu_6=1.0`
2. Compare a previsao com o BVAR Minnesota padrao
3. A prior Sims-Zha produz previsoes mais ou menos dispersas?

In [ ]:
# --- Exercicio 2: Seu codigo aqui ---

# Dica:
# bvar_sz = BayesianVAR(lags=4, prior="sims_zha", lambda_1=0.1, mu_5=1.0, mu_6=1.0)
# res_sz = bvar_sz.fit(endog, n_draws=3000, burnin=500, seed=42)
# fc_sz = res_sz.forecast(steps=12)
# print(f"Sims-Zha log ML: {res_sz.log_marginal_likelihood:.2f}")

---

## Resumo

Neste notebook aprendemos:

1. **Motivacao para BVAR**: VARs frequentistas sofrem da maldição da dimensionalidade.
   Priors bayesianos regularizam as estimativas via shrinkage.

2. **Minnesota prior**: encolhe os coeficientes na direcao de um random walk, com
   hyperparametros controlando a forca do shrinkage ($\lambda_1$, $\lambda_2$, $\lambda_3$).

3. **Forma reduzida vs estrutural no BVAR**: o BVAR estima a forma reduzida via
   posterior. IRFs estruturais sao obtidas aplicando Cholesky sobre a covariancia
   posterior, gerando bandas crediveis via draws do posterior.

4. **Previsao bayesiana**: intervalos de credibilidade incorporam tanto a incerteza
   dos parametros quanto a incerteza dos choques futuros.

5. **Selecao de modelos**: a marginal likelihood permite comparar priors e
   hyperparametros de forma principiada (Bayes factors).

6. **BVAR vs VAR frequentista**: o BVAR produz IRFs mais suaves e previsoes com
   incerteza calibrada. E especialmente vantajoso em amostras pequenas ou modelos
   grandes.